Model Comparison
Purpose
The purpose of this notebook is to compare multiple machine learning algorithms and identify the model that delivers the strongest customer churn prediction performance.

Objectives
Train multiple classification algorithms
Compare model performance
Evaluate prediction quality
Identify strengths and weaknesses of each model
Select a candidate model for hyperparameter tuning
The best-performing model will be carried forward into the final pipeline and optimisation stages.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier)

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

print("Libraries Loaded")

Libraries Loaded


Load and Prepare Dataset
The same preprocessing workflow used during baseline modelling will be applied to ensure fair model comparisons.

In [2]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.dropna(subset=["TotalCharges"])

df = df.drop("customerID", axis=1)

X = df.drop("Churn", axis=1)

y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Dataset Ready")

Dataset Ready


Build Preprocessing Pipeline
The same preprocessing pipeline will be reused across every model to ensure that all algorithms receive identical input data.

In [3]:
numerical_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

categorical_features = [col for col in X.columns if col not in numerical_features]


In [4]:
preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

Define Candidate Models
The following models will be evaluated:

Logistic Regression
Decision Tree
Random Forest
Gradient Boosting
These models represent increasing levels of complexity and predictive capability.

In [5]:
models = {"Logistic Regression": LogisticRegression(random_state=42), 
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42)}

Train and Evaluate Models
Each model will be trained using the same dataset and evaluated using identical performance metrics.

This ensures a fair comparison between algorithms.

In [6]:
results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({"Model": name, "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label="Yes"),
        "Recall": recall_score(y_test, y_pred, pos_label="Yes"),
        "F1 Score": f1_score(y_test, y_pred, pos_label="Yes"),
        "ROC-AUC": roc_auc_score((y_test == "Yes").astype(int),y_prob)})

print("Model Training Complete")

Model Training Complete


Model Comparison Results
The performance metrics for each model will now be collected into a single comparison table.

This comparison will help identify the strongest candidate model for further optimisation and final deployment.

In [7]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.803838,0.648485,0.572193,0.607955,0.835929
1,Decision Tree,0.730633,0.493369,0.497326,0.495340,0.656455
2,Random Forest,0.787491,0.630662,0.483957,0.547655,0.813661
3,Gradient Boosting,0.796731,0.642857,0.529412,0.580645,0.838614


In [8]:
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

results_df.reset_index(drop=True, inplace=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Gradient Boosting,0.796731,0.642857,0.529412,0.580645,0.838614
1,Logistic Regression,0.803838,0.648485,0.572193,0.607955,0.835929
2,Random Forest,0.787491,0.630662,0.483957,0.547655,0.813661
3,Decision Tree,0.730633,0.493369,0.497326,0.495340,0.656455


Findings
The model comparison identified clear differences in predictive performance across the evaluated algorithms.

Model Ranking
Based on ROC-AUC performance, the models ranked as follows:

Gradient Boosting
Logistic Regression
Random Forest
Decision Tree
Key Observations
Gradient Boosting
Gradient Boosting achieved the highest ROC-AUC score and delivered the strongest overall predictive performance.

The model appears capable of capturing complex patterns and interactions within the data, providing a slight performance advantage over the other algorithms.

Logistic Regression
Logistic Regression produced performance that was remarkably close to Gradient Boosting.

Despite being a considerably simpler model, it achieved very strong results across all evaluation metrics and demonstrated excellent predictive capability.

Random Forest
Random Forest achieved good overall performance but did not outperform the top two models.

Decision Tree
Decision Tree produced the weakest results and demonstrated substantially lower performance across every evaluation metric.

Business Interpretation
The results demonstrate that customer churn can be predicted effectively using the available customer information.

The strongest-performing models successfully leverage patterns associated with:

Contract type
Customer tenure
Internet service type
Monthly charges
These findings align closely with the insights discovered during exploratory data analysis.

Model Selection Considerations
While Gradient Boosting achieved the highest performance, the difference between Gradient Boosting and Logistic Regression is relatively small.

This suggests that both models should be considered during final model selection:

Gradient Boosting offers the highest predictive performance.
Logistic Regression offers greater simplicity and interpretability.
Both models will be carried forward into the final modelling phase.

Summary and Key Findings
The model comparison phase successfully evaluated multiple machine learning algorithms for customer churn prediction.

Models Evaluated
Logistic Regression
Decision Tree
Random Forest
Gradient Boosting
Performance Summary
Model	Accuracy	Precision	Recall	F1 Score	ROC-AUC
Gradient Boosting	0.7967	0.6429	0.5294	0.5806	0.8386
Logistic Regression	0.8038	0.6485	0.5722	0.6080	0.8359
Random Forest	0.7875	0.6307	0.4840	0.5477	0.8137
Decision Tree	0.7306	0.4934	0.4973	0.4953	0.6565
Best Performing Models
Gradient Boosting
Strengths:

Highest ROC-AUC
Strong overall predictive performance
Captures complex feature interactions
Logistic Regression
Strengths:

Highest accuracy
Highest F1 Score
Highest recall
Easy to interpret
Computationally efficient
Key Insights
The results indicate that:

Ensemble methods outperform a single Decision Tree.
Customer churn contains both linear and non-linear patterns.
Logistic Regression provides a very strong baseline.
Gradient Boosting delivers the strongest overall discrimination capability.
Recommended Candidates
Two models will proceed to the final stage:

Gradient Boosting
Logistic Regression
These models provide the best balance between predictive performance and interpretability.

Next Steps
The final notebook will focus on:

Final model selection
Hyperparameter tuning
Pipeline development
Model persistence
Production readiness
The objective is to build a complete machine learning pipeline capable of generating customer churn predictions from raw customer information.